In [ ]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import AdaBoostClassifier, BaggingClassifier, RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from catboost import CatBoostClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, precision_score, recall_score
import time
import shap
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.decomposition import PCA
import matplotlib.pyplot  as plt

In [25]:
class OutlierHandler(BaseEstimator, TransformerMixin):
    def __init__(self, factor=1.5):
        self.factor = factor
        self.lower_bounds = {}
        self.upper_bounds = {}

    def fit(self, X, y=None):
        X_df = pd.DataFrame(X)
        for col in X_df.columns:
            Q1 = X_df[col].quantile(0.25)
            Q3 = X_df[col].quantile(0.75)
            IQR = Q3 - Q1
            self.lower_bounds[col] = Q1 - self.factor * IQR
            self.upper_bounds[col] = Q3 + self.factor * IQR
        return self

    def transform(self, X, y=None):
        X_df = pd.DataFrame(X).copy()
        for col in X_df.columns:
            # Capping (Winsorization)
            X_df[col] = np.where(X_df[col] < self.lower_bounds[col], self.lower_bounds[col], X_df[col])
            X_df[col] = np.where(X_df[col] > self.upper_bounds[col], self.upper_bounds[col], X_df[col])
        return X_df.values

In [26]:
df = pd.read_csv('./dataset/larxel_kg/heart.csv')

In [27]:
df.duplicated().sum()

np.int64(0)

In [28]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 299 entries, 0 to 298
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   age                       299 non-null    float64
 1   anaemia                   299 non-null    int64  
 2   creatinine_phosphokinase  299 non-null    int64  
 3   diabetes                  299 non-null    int64  
 4   ejection_fraction         299 non-null    int64  
 5   high_blood_pressure       299 non-null    int64  
 6   platelets                 299 non-null    float64
 7   serum_creatinine          299 non-null    float64
 8   serum_sodium              299 non-null    int64  
 9   sex                       299 non-null    int64  
 10  smoking                   299 non-null    int64  
 11  time                      299 non-null    int64  
 12  DEATH_EVENT               299 non-null    int64  
dtypes: float64(3), int64(10)
memory usage: 30.5 KB


In [29]:
df['DEATH_EVENT'].value_counts()

DEATH_EVENT
0    203
1     96
Name: count, dtype: int64

In [30]:
df['DEATH_EVENT'] = np.where(df['DEATH_EVENT'] == 0, 0, 1)

In [31]:
df.drop('time',axis=1)

,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,DEATH_EVENT
0,75.0,0,582,0,20,1,265000.00,1.9,130,1,0,1
1,55.0,0,7861,0,38,0,263358.03,1.1,136,1,0,1
2,65.0,0,146,0,20,0,162000.00,1.3,129,1,1,1
3,50.0,1,111,0,20,0,210000.00,1.9,137,1,0,1
4,65.0,1,160,1,20,0,327000.00,2.7,116,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...
294,62.0,0,61,1,38,1,155000.00,1.1,143,1,1,0
295,55.0,0,1820,0,38,0,270000.00,1.2,139,0,0,0
296,45.0,0,2060,1,60,0,742000.00,0.8,138,0,0,0
297,45.0,0,2413,0,38,0,140000.00,1.4,140,1,1,0


In [ ]:
X = df.drop('DEATH_EVENT', axis=1)
y = df['DEATH_EVENT']

X_train,X_test,Y_train,Y_test = train_test_split(X,y,test_size=0.2,stratify=y,random_state=42)


num_features = [
    "age",                         # tuổi
    "creatinine_phosphokinase",    # enzyme CPK
    "ejection_fraction",           # EF (%)
    "platelets",                   # tiểu cầu
    "serum_creatinine",            # creatinine máu
    "serum_sodium",                # natri máu
]
cat_features = [
    "anaemia",
    "diabetes",
    "high_blood_pressure",
    "sex",
    "smoking"
    ]

In [33]:
Y_train.value_counts()

DEATH_EVENT
0    162
1     77
Name: count, dtype: int64

In [34]:
Y_test.value_counts()

DEATH_EVENT
0    41
1    19
Name: count, dtype: int64

PREPROCESSING DATA KNN-SVM-LR

In [35]:
num_pipeline_scaled = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("outlier", OutlierHandler()),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocess_scaled = ColumnTransformer([
    ("num", num_pipeline_scaled, num_features),
    ("cat", cat_pipeline, cat_features)
])

Decision Tree, Random Forest, Bagging

In [36]:
num_pipeline_no_scale = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

preprocess_tree = ColumnTransformer([
    ("num", num_pipeline_no_scale, num_features),
    ("cat", cat_pipeline, cat_features)
])

XGBoost, AdaBoost

In [37]:
preprocess_boost = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ]), num_features),
    
    ("cat", cat_pipeline, cat_features)
])

In [38]:
CatBoostClassifier(cat_features=cat_features)

CatBoostClassifier(cat_features=['anaemia', 'diabetes', 'high_blood_pressure', 'sex', 'smoking'])

In [39]:
pipe_knn = Pipeline([
    ("preprocess", preprocess_scaled),
    ("model", KNeighborsClassifier())
])

pipe_lr = Pipeline([
    ("preprocess", preprocess_scaled),
    ("model", LogisticRegression(max_iter=1000))
])

pipe_svm = Pipeline([
    ("preprocess", preprocess_scaled),
    ("model", SVC(probability=True))
])

pipe_dt = Pipeline([
    ("preprocess", preprocess_tree),
    ("model", DecisionTreeClassifier())
])

pipe_rf = Pipeline([
    ("preprocess", preprocess_tree),
    ("model", RandomForestClassifier())
])

pipe_bagging = Pipeline([
    ("preprocess", preprocess_tree),
    ("model", BaggingClassifier(
        estimator=DecisionTreeClassifier(),
        n_estimators=100
    ))
])

pipe_xgb = Pipeline([
    ("preprocess", preprocess_boost),
    ("model", XGBClassifier(eval_metric='logloss'))
])

pipe_ada = Pipeline([
    ("preprocess", preprocess_boost),
    ("model", AdaBoostClassifier())
])

pipe_cat = Pipeline([
    ("preprocess", preprocess_boost),
    ("model", CatBoostClassifier(verbose=0))
])

In [40]:
pipelines = {
    "KNN": pipe_knn,
    "LR": pipe_lr,
    "SVM": pipe_svm,
    "DT": pipe_dt,
    "RF": pipe_rf,
    "Bagging": pipe_bagging,
    "XGB": pipe_xgb,
    "Ada": pipe_ada,
    "CatBoost": pipe_cat
}

In [41]:
param_knn = {
    "model__n_neighbors": [3,5,7,9,11],
    "model__weights": ["uniform", "distance"],
    "model__p": [1, 2]
}

param_lr = {
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__penalty": ["l2"],
    "model__solver": ["lbfgs"]
}

param_svm = {
    "model__C": [0.1, 1, 10, 100],
    "model__kernel": ["rbf", "linear"],
    "model__gamma": ["scale", "auto"]
}

param_dt = {
    "model__max_depth": [3,5,10,20,None],
    "model__min_samples_split": [2,5,10],
    "model__min_samples_leaf": [1,2,4],
    "model__criterion": ["gini", "entropy"]
}

param_rf = {
    "model__n_estimators": [100,200,300],
    "model__max_depth": [None,5,10,20],
    "model__min_samples_split": [2,5],
    "model__min_samples_leaf": [1,2],
    "model__max_features": ["sqrt", "log2"]
}

param_bagging = {
    "model__n_estimators": [50,100,200],
    "model__max_samples": [0.5, 0.7, 1.0],
    "model__max_features": [0.5, 0.7, 1.0]
}

param_xgb = {
    "model__n_estimators": [100,200,300],
    "model__max_depth": [3,5,7],
    "model__learning_rate": [0.01, 0.1, 0.2],
    "model__subsample": [0.8, 1.0],
    "model__colsample_bytree": [0.8, 1.0]
}

param_ada = {
    "model__n_estimators": [50,100,200],
    "model__learning_rate": [0.01, 0.1, 1]
}

param_cat = {
    "model__iterations": [100,200,300],
    "model__depth": [4,6,8],
    "model__learning_rate": [0.01, 0.1],
    "model__l2_leaf_reg": [1,3,5]
}

In [42]:
param_grids = {
    "KNN": param_knn,
    "LR": param_lr,
    "SVM": param_svm,
    "DT": param_dt,
    "RF": param_rf,
    "Bagging": param_bagging,
    "XGB": param_xgb,
    "Ada": param_ada,
    "CatBoost": param_cat
}

In [ ]:
best_models = {}
evaluation_results = []

for name, pipe in pipelines.items():
    print(f"[{name}] Đang huấn luyện với GridSearchCV...")
    if name == 'SVM' or name == 'Bagging' or name == 'CatBoost':
        continue
    # 1. Huấn luyện GridSearchCV
    grid = GridSearchCV(
        pipe,
        param_grids[name],
        cv=5,
        scoring="roc_auc",
        n_jobs=-1,
        return_train_score=True,
        verbose=1
    )
    
    start_time = time.time()
    
    grid.fit(X_train, Y_train)
    
    end_time = time.time()
    training_time = end_time - start_time
    
    best_models[name] = grid.best_estimator_
    
    
    best_idx = grid.best_index_
    
    # roc_aucs trả về số âm, nên ta nhân -1 để có giá trị log_loss chuẩn (càng nhỏ càng tốt)
    train_loss = -grid.cv_results_['mean_train_score'][best_idx]
    val_loss_cv = -grid.cv_results_['mean_test_score'][best_idx]
    
    # 2. Dự đoán xác suất cho nhãn Positive (1)
    # SVM của bạn đã có probability=True nên predict_proba sẽ chạy bình thường
    y_probs = grid.best_estimator_.predict_proba(X_test)[:, 1]
    
    # 3. Tính toán ROC AUC và tìm ngưỡng cắt (Threshold) tối ưu
    auc = roc_auc_score(Y_test, y_probs)
    fpr, tpr, thresholds = roc_curve(Y_test, y_probs)
    
    # Sử dụng Youden's J statistic để tìm threshold cắt tối ưu (tối đa hoá tpr - fpr)
    J = tpr - fpr
    optimal_idx = np.argmax(J)
    optimal_threshold = thresholds[optimal_idx]
    
    # 4. Dự đoán dựa trên ngưỡng vừa tìm được
    y_pred = (y_probs >= optimal_threshold).astype(int)
    
    # 5. Đánh giá các độ đo khác
    acc = accuracy_score(Y_test, y_pred)
    prec = precision_score(Y_test, y_pred, zero_division=0)
    rec = recall_score(Y_test, y_pred, zero_division=0)
    
    evaluation_results.append({
        "Model": name,
        "Training Time (s)": round(training_time, 4),
        "AUC": auc,
        "Optimal Threshold": optimal_threshold,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Best Params": grid.best_params_
    })
    print(grid.best_params_)
# Tạo DataFrame hiển thị kết quả
results_df = pd.DataFrame(evaluation_results)
results_df.sort_values(by="AUC", ascending=False, inplace=True)
results_df.reset_index(drop=True, inplace=True)

display(results_df)

[KNN] Đang huấn luyện với GridSearchCV...
Fitting 5 folds for each of 20 candidates, totalling 100 fits
{'model__n_neighbors': 11, 'model__p': 2, 'model__weights': 'distance'}
[LR] Đang huấn luyện với GridSearchCV...
Fitting 5 folds for each of 5 candidates, totalling 25 fits


c:\Users\admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


{'model__C': 0.01, 'model__penalty': 'l2', 'model__solver': 'lbfgs'}
[SVM] Đang huấn luyện với GridSearchCV...
Fitting 5 folds for each of 16 candidates, totalling 80 fits
{'model__C': 0.1, 'model__gamma': 'auto', 'model__kernel': 'rbf'}
[DT] Đang huấn luyện với GridSearchCV...
Fitting 5 folds for each of 90 candidates, totalling 450 fits
{'model__criterion': 'entropy', 'model__max_depth': 3, 'model__min_samples_leaf': 4, 'model__min_samples_split': 2}
[RF] Đang huấn luyện với GridSearchCV...
Fitting 5 folds for each of 96 candidates, totalling 480 fits
{'model__max_depth': 5, 'model__max_features': 'log2', 'model__min_samples_leaf': 1, 'model__min_samples_split': 5, 'model__n_estimators': 300}
[Bagging] Đang huấn luyện với GridSearchCV...
Fitting 5 folds for each of 27 candidates, totalling 135 fits
{'model__max_features': 0.5, 'model__max_samples': 0.5, 'model__n_estimators': 200}
[XGB] Đang huấn luyện với GridSearchCV...
Fitting 5 folds for each of 108 candidates, totalling 540 fits

,Model,Training Time (s),AUC,Optimal Threshold,Accuracy,Precision,Recall,Best Params
0,SVM,3.4320,0.843389,0.606658,0.833333,0.800000,0.631579,"{'model__C': 0.1, 'model__gamma': 'auto', 'mod..."
1,RF,47.2773,0.813864,0.200654,0.733333,0.551724,0.842105,"{'model__max_depth': 5, 'model__max_features':..."
2,CatBoost,118.0913,0.808729,0.381859,0.766667,0.608696,0.736842,"{'model__depth': 4, 'model__iterations': 100, ..."
3,Bagging,11.9278,0.790757,0.337500,0.733333,0.555556,0.789474,"{'model__max_features': 0.5, 'model__max_sampl..."
4,XGB,11.9364,0.784339,0.284300,0.800000,0.666667,0.736842,"{'model__colsample_bytree': 0.8, 'model__learn..."
5,LR,0.3797,0.776637,0.403713,0.833333,0.800000,0.631579,"{'model__C': 0.01, 'model__penalty': 'l2', 'mo..."
6,Ada,7.5839,0.750963,0.293891,0.766667,0.608696,0.736842,"{'model__learning_rate': 0.1, 'model__n_estima..."
7,DT,5.5426,0.730424,0.484848,0.766667,0.619048,0.684211,"{'model__criterion': 'entropy', 'model__max_de..."
8,KNN,11.8816,0.727856,0.437511,0.833333,0.800000,0.631579,"{'model__n_neighbors': 11, 'model__p': 2, 'mod..."


In [ ]:
if 'LR' in logistic_regression_models:
    lr_data = logistic_regression_models['LR']
    
    # Tạo DataFrame chi tiết
    lr_coefficients_df = pd.DataFrame({
        "Feature": lr_data["features"],
        "Coefficient": lr_data["coefficients"],
        "Abs_Coefficient": np.abs(lr_data["coefficients"]),
        "Impact": ["Positive" if coef > 0 else "Negative" for coef in lr_data["coefficients"]]
    })
    
    # Sắp xếp theo giá trị tuyệt đối giảm dần
    lr_coefficients_df = lr_coefficients_df.sort_values("Abs_Coefficient", ascending=False)
    lr_coefficients_df = lr_coefficients_df.round(4)
    
    print("\n" + "="*80)
    print("BẢNG HỆ SỐ HỒI QUY LOGISTIC REGRESSION")
    print("="*80)
    print(f"\nBest Parameters: {lr_data['best_params']}")
    print(f"AUC Score: {lr_data['auc']:.4f}")
    print("\n")
    
    # Hiển thị bảng
    from IPython.display import display
    display(lr_coefficients_df)
    
    # Lưu ra file CSV nếu muốn
    lr_coefficients_df.to_csv("logistic_regression_coefficients.csv", index=False)
    print("\n✅ Đã lưu file: logistic_regression_coefficients.csv")
    
    # ============================================
    # THỐNG KÊ TÓM TẮT
    # ============================================
    print("\n" + "="*80)
    print("THỐNG KÊ TÓM TẮT")
    print("="*80)
    
    positive_impact = lr_coefficients_df[lr_coefficients_df["Impact"] == "Positive"]
    negative_impact = lr_coefficients_df[lr_coefficients_df["Impact"] == "Negative"]
    
    print(f"\n📈 Features có tác động TÍCH CỰC (tăng nguy cơ bệnh): {len(positive_impact)} features")
    print(positive_impact[["Feature", "Coefficient"]].to_string(index=False))
    
    print(f"\n📉 Features có tác động TIÊU CỰC (giảm nguy cơ bệnh): {len(negative_impact)} features")
    print(negative_impact[["Feature", "Coefficient"]].to_string(index=False))
    
    # ============================================
    # VẼ BIỂU ĐỒ (tùy chọn)
    # ============================================
    plt.figure(figsize=(12, 7))
    colors = ['green' if coef > 0 else 'red' for coef in lr_coefficients_df["Coefficient"]]
    plt.barh(lr_coefficients_df["Feature"], lr_coefficients_df["Coefficient"], color=colors)
    plt.axvline(x=0, color='black', linestyle='-', linewidth=0.8)
    plt.xlabel("Coefficient Value", fontsize=12)
    plt.ylabel("Features", fontsize=12)
    plt.title("Logistic Regression Coefficients - Feature Importance", fontsize=14)
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ Không tìm thấy mô hình Logistic Regression trong best_models")

In [ ]:
best_lr = best_models["LR"]  # Pipeline

preprocessor = best_lr.named_steps["preprocess"]
num_features_names = num_features
cat_encoded = (preprocessor.named_transformers_["cat"]
               .named_steps["encoder"]
               .get_feature_names_out(cat_features).tolist())
all_features = num_features_names + cat_encoded

lr_model = best_lr.named_steps["model"]
coef_abs = np.abs(lr_model.coef_[0])

import matplotlib.pyplot as plt

importance_df = pd.DataFrame({
    "Feature": all_features,
    "Importance": coef_abs
}).sort_values("Importance", ascending=False)

# Vẽ
plt.figure(figsize=(10, 6))
plt.barh(importance_df["Feature"], importance_df["Importance"], color="steelblue")
plt.gca().invert_yaxis()
plt.title("Feature Importance - Logistic Regression", fontsize=14)
plt.xlabel("Absolute Coefficient")
plt.tight_layout()
plt.show()

In [44]:
smote_step = SMOTE(sampling_strategy=0.7)

pipe_knn = ImbPipeline([
    ("preprocess", preprocess_scaled),
    ("smote", smote_step),
    ("model", KNeighborsClassifier())
])

pipe_lr = ImbPipeline([
    ("preprocess", preprocess_scaled),
    ("smote", smote_step),
    ("model", LogisticRegression())
])

pipe_svm = ImbPipeline([
    ("preprocess", preprocess_scaled),
    ("smote", smote_step),
    ("model", SVC(probability=True))
])

pipe_dt = ImbPipeline([
    ("preprocess", preprocess_tree),
    ("smote", smote_step),
    ("model", DecisionTreeClassifier())
])

pipe_rf = ImbPipeline([
    ("preprocess", preprocess_tree),
    ("smote", smote_step),
    ("model", RandomForestClassifier())
])

pipe_bagging = ImbPipeline([
    ("preprocess", preprocess_tree),
    ("smote", smote_step),
    ("model", BaggingClassifier(
        estimator=DecisionTreeClassifier(),
        n_estimators=100
    ))
])

pipe_xgb = ImbPipeline([
    ("preprocess", preprocess_boost),
    ("smote", smote_step),
    ("model", XGBClassifier(eval_metric='logloss'))
])

pipe_ada = ImbPipeline([
    ("preprocess", preprocess_boost),
    ("smote", smote_step),
    ("model", AdaBoostClassifier())
])

pipe_cat = ImbPipeline([
    ("preprocess", preprocess_boost),
    ("smote", smote_step),
    ("model", CatBoostClassifier(verbose=0))
])

In [45]:
pipelines_smote = {
    "KNN": pipe_knn,
    "LR": pipe_lr,
    "SVM": pipe_svm,
    "DT": pipe_dt,
    "RF": pipe_rf,
    "Bagging": pipe_bagging,
    "XGB": pipe_xgb,
    "Ada": pipe_ada,
    "CatBoost": pipe_cat
}

In [ ]:
best_models = {}
evaluation_results = []

for name, pipe in pipelines_smote.items():
    if name == 'SVM' or name == 'Bagging' or name == 'CatBoost':
        continue
    print(f"[{name}] Đang huấn luyện với GridSearchCV...")
    
    # 1. Huấn luyện GridSearchCV
    grid = GridSearchCV(
        pipe,
        param_grids[name],
        cv=5,
        scoring="roc_auc",
        n_jobs=-1,
        return_train_score=True,
        verbose=1
    )
    
    start_time = time.time()
    
    grid.fit(X_train, Y_train)
    
    end_time = time.time()
    training_time = end_time - start_time
    
    best_models[name] = grid.best_estimator_
    
    
    best_idx = grid.best_index_
    
    # roc_aucs trả về số âm, nên ta nhân -1 để có giá trị log_loss chuẩn (càng nhỏ càng tốt)
    train_loss = -grid.cv_results_['mean_train_score'][best_idx]
    val_loss_cv = -grid.cv_results_['mean_test_score'][best_idx]
    
    # 2. Dự đoán xác suất cho nhãn Positive (1)
    # SVM của bạn đã có probability=True nên predict_proba sẽ chạy bình thường
    y_probs = grid.best_estimator_.predict_proba(X_test)[:, 1]
    
    # 3. Tính toán ROC AUC và tìm ngưỡng cắt (Threshold) tối ưu
    auc = roc_auc_score(Y_test, y_probs)
    fpr, tpr, thresholds = roc_curve(Y_test, y_probs)
    
    # Sử dụng Youden's J statistic để tìm threshold cắt tối ưu (tối đa hoá tpr - fpr)
    J = tpr - fpr
    optimal_idx = np.argmax(J)
    optimal_threshold = thresholds[optimal_idx]
    
    # 4. Dự đoán dựa trên ngưỡng vừa tìm được
    y_pred = (y_probs >= optimal_threshold).astype(int)
    
    # 5. Đánh giá các độ đo khác
    acc = accuracy_score(Y_test, y_pred)
    prec = precision_score(Y_test, y_pred, zero_division=0)
    rec = recall_score(Y_test, y_pred, zero_division=0)
    
    evaluation_results.append({
        "Model": name,
        "Training Time (s)": round(training_time, 4),
        "AUC": auc,
        "Optimal Threshold": optimal_threshold,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Best Params": grid.best_params_
    })
    print(grid.best_params_)
# Tạo DataFrame hiển thị kết quả
results_df = pd.DataFrame(evaluation_results)
results_df.sort_values(by="AUC", ascending=False, inplace=True)
results_df.reset_index(drop=True, inplace=True)

display(results_df)

[KNN] Đang huấn luyện với GridSearchCV...
Fitting 5 folds for each of 20 candidates, totalling 100 fits
{'model__n_neighbors': 11, 'model__p': 1, 'model__weights': 'distance'}
[LR] Đang huấn luyện với GridSearchCV...
Fitting 5 folds for each of 5 candidates, totalling 25 fits


c:\Users\admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


{'model__C': 10, 'model__penalty': 'l2', 'model__solver': 'lbfgs'}
[SVM] Đang huấn luyện với GridSearchCV...
Fitting 5 folds for each of 16 candidates, totalling 80 fits
{'model__C': 0.1, 'model__gamma': 'scale', 'model__kernel': 'rbf'}
[DT] Đang huấn luyện với GridSearchCV...
Fitting 5 folds for each of 90 candidates, totalling 450 fits
{'model__criterion': 'entropy', 'model__max_depth': 3, 'model__min_samples_leaf': 4, 'model__min_samples_split': 10}
[RF] Đang huấn luyện với GridSearchCV...
Fitting 5 folds for each of 96 candidates, totalling 480 fits
{'model__max_depth': 20, 'model__max_features': 'log2', 'model__min_samples_leaf': 2, 'model__min_samples_split': 5, 'model__n_estimators': 100}
[Bagging] Đang huấn luyện với GridSearchCV...
Fitting 5 folds for each of 27 candidates, totalling 135 fits
{'model__max_features': 1.0, 'model__max_samples': 0.5, 'model__n_estimators': 200}
[XGB] Đang huấn luyện với GridSearchCV...
Fitting 5 folds for each of 108 candidates, totalling 540 fit

,Model,Training Time (s),AUC,Optimal Threshold,Accuracy,Precision,Recall,Best Params
0,SVM,5.7519,0.824134,0.665422,0.833333,0.800000,0.631579,"{'model__C': 0.1, 'model__gamma': 'scale', 'mo..."
1,RF,99.0004,0.810013,0.507083,0.800000,0.705882,0.631579,"{'model__max_depth': 20, 'model__max_features'..."
2,CatBoost,111.6074,0.807445,0.141728,0.683333,0.500000,0.947368,"{'model__depth': 4, 'model__iterations': 100, ..."
3,XGB,26.1740,0.802311,0.417621,0.783333,0.636364,0.736842,"{'model__colsample_bytree': 0.8, 'model__learn..."
4,KNN,4.0142,0.800385,0.411044,0.800000,0.666667,0.736842,"{'model__n_neighbors': 11, 'model__p': 1, 'mod..."
5,Bagging,22.8026,0.792683,0.605000,0.783333,0.666667,0.631579,"{'model__max_features': 1.0, 'model__max_sampl..."
6,Ada,6.7382,0.781130,0.438283,0.800000,0.666667,0.736842,"{'model__learning_rate': 0.1, 'model__n_estima..."
7,LR,1.1020,0.768935,0.610560,0.816667,0.750000,0.631579,"{'model__C': 10, 'model__penalty': 'l2', 'mode..."
8,DT,8.4695,0.688703,0.375000,0.750000,0.611111,0.578947,"{'model__criterion': 'entropy', 'model__max_de..."


In [47]:

pca_step = PCA() # Để trống tham số này để Param Grid tự tune

# TẤT CẢ các model đều phải dùng preprocess_scaled vì có PCA
pipe_knn = ImbPipeline([
    ("preprocess", preprocess_scaled),
    ("smote", smote_step),
    ("pca", pca_step),
    ("model", KNeighborsClassifier())
])

pipe_lr = ImbPipeline([
    ("preprocess", preprocess_scaled),
    ("smote", smote_step),
    ("pca", pca_step),
    ("model", LogisticRegression())
])

pipe_svm = ImbPipeline([
    ("preprocess", preprocess_scaled),
    ("smote", smote_step),
    ("pca", pca_step),
    ("model", SVC(probability=True))
])

pipe_dt = ImbPipeline([
    ("preprocess", preprocess_scaled),
    ("smote", smote_step),
    ("pca", pca_step),
    ("model", DecisionTreeClassifier())
])

pipe_rf = ImbPipeline([
    ("preprocess", preprocess_scaled),
    ("smote", smote_step),
    ("pca", pca_step),
    ("model", RandomForestClassifier())
])

pipe_bagging = ImbPipeline([
    ("preprocess", preprocess_scaled),
    ("smote", smote_step),
    ("pca", pca_step),
    ("model", BaggingClassifier(
        estimator=DecisionTreeClassifier(),
        n_estimators=100
    ))
])

pipe_xgb = ImbPipeline([
    ("preprocess", preprocess_scaled),
    ("smote", smote_step),
    ("pca", pca_step),
    ("model", XGBClassifier(eval_metric='logloss'))
])

pipe_ada = ImbPipeline([
    ("preprocess", preprocess_scaled),
    ("smote", smote_step),
    ("pca", pca_step),
    ("model", AdaBoostClassifier())
])

pipe_cat = ImbPipeline([
    ("preprocess", preprocess_scaled),
    ("smote", smote_step),
    ("pca", pca_step),
    ("model", CatBoostClassifier(verbose=0))
])

pipelines_smote_pca = {
    "KNN": pipe_knn,
    "LR": pipe_lr,
    "SVM": pipe_svm,
    "DT": pipe_dt,
    "RF": pipe_rf,
    "Bagging": pipe_bagging,
    "XGB": pipe_xgb,
    "Ada": pipe_ada,
    "CatBoost": pipe_cat
}

In [48]:
param_knn = {
    "pca__n_components": [0.85, 0.90, 0.95],
    "model__n_neighbors": [3, 5, 7, 9, 11],
    "model__weights": ["uniform", "distance"],
    "model__p": [1, 2]
}

param_lr = {
    "pca__n_components": [0.85, 0.90, 0.95],
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__penalty": ["l2"],
    "model__solver": ["lbfgs"]
}

param_svm = {
    "pca__n_components": [0.85, 0.90, 0.95],
    "model__C": [0.1, 1, 10, 100],
    "model__kernel": ["rbf", "linear"],
    "model__gamma": ["scale", "auto"]
}

param_dt = {
    "pca__n_components": [0.85, 0.90, 0.95],
    "model__max_depth": [3, 5, 10, 20, None],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__criterion": ["gini", "entropy"]
}

param_rf = {
    "pca__n_components": [0.85, 0.90, 0.95],
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [None, 5, 10, 20],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2]
}

param_bagging = {
    "pca__n_components": [0.85, 0.90, 0.95],
    "model__n_estimators": [50, 100, 200],
    "model__max_samples": [0.5, 0.7, 1.0],
    "model__max_features": [0.5, 0.7, 1.0]
}

param_xgb = {
    "pca__n_components": [0.85, 0.90, 0.95],
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [3, 5, 7],
    "model__learning_rate": [0.01, 0.1, 0.2],
    "model__subsample": [0.8, 1.0],
    "model__colsample_bytree": [0.8, 1.0]
}

param_ada = {
    "pca__n_components": [0.85, 0.90, 0.95],
    "model__n_estimators": [50, 100, 200],
    "model__learning_rate": [0.01, 0.1, 1]
}

param_cat = {
    "pca__n_components": [0.85, 0.90, 0.95],
    "model__iterations": [100, 200, 300],
    "model__depth": [4, 6, 8],
    "model__learning_rate": [0.01, 0.1]
    # Bỏ l2_leaf_reg đi bớt để giảm thời gian train nếu bạn kết hợp cả PCA
}

param_grids_smote_pca = {
    "KNN": param_knn,
    "LR": param_lr,
    "SVM": param_svm,
    "DT": param_dt,
    "RF": param_rf,
    "Bagging": param_bagging,
    "XGB": param_xgb,
    "Ada": param_ada,
    "CatBoost": param_cat
}

In [ ]:
best_models = {}
evaluation_results = []

for name, pipe in pipelines_smote_pca.items():
    if name == 'SVM' or name == 'Bagging' or name == 'CatBoost':
        continue
    print(f"[{name}] Đang huấn luyện với GridSearchCV...")
    
    # 1. Huấn luyện GridSearchCV
    grid = GridSearchCV(
        pipe,
        param_grids_smote_pca[name],
        cv=5,
        scoring="roc_auc",
        n_jobs=-1,
        return_train_score=True,
        verbose=1
    )
    
    start_time = time.time()
    
    grid.fit(X_train, Y_train)
    
    end_time = time.time()
    training_time = end_time - start_time
    
    best_models[name] = grid.best_estimator_
    
    
    best_idx = grid.best_index_
    
    # roc_aucs trả về số âm, nên ta nhân -1 để có giá trị log_loss chuẩn (càng nhỏ càng tốt)
    train_loss = -grid.cv_results_['mean_train_score'][best_idx]
    val_loss_cv = -grid.cv_results_['mean_test_score'][best_idx]
    
    # 2. Dự đoán xác suất cho nhãn Positive (1)
    # SVM của bạn đã có probability=True nên predict_proba sẽ chạy bình thường
    y_probs = grid.best_estimator_.predict_proba(X_test)[:, 1]
    
    # 3. Tính toán ROC AUC và tìm ngưỡng cắt (Threshold) tối ưu
    auc = roc_auc_score(Y_test, y_probs)
    fpr, tpr, thresholds = roc_curve(Y_test, y_probs)
    
    # Sử dụng Youden's J statistic để tìm threshold cắt tối ưu (tối đa hoá tpr - fpr)
    J = tpr - fpr
    optimal_idx = np.argmax(J)
    optimal_threshold = thresholds[optimal_idx]
    
    # 4. Dự đoán dựa trên ngưỡng vừa tìm được
    y_pred = (y_probs >= optimal_threshold).astype(int)
    
    # 5. Đánh giá các độ đo khác
    acc = accuracy_score(Y_test, y_pred)
    prec = precision_score(Y_test, y_pred, zero_division=0)
    rec = recall_score(Y_test, y_pred, zero_division=0)
    
    evaluation_results.append({
        "Model": name,
        "Training Time (s)": round(training_time, 4),
        "AUC": auc,
        "Optimal Threshold": optimal_threshold,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Best Params": grid.best_params_
    })
    print(grid.best_params_)
# Tạo DataFrame hiển thị kết quả
results_df = pd.DataFrame(evaluation_results)
results_df.sort_values(by="AUC", ascending=False, inplace=True)
results_df.reset_index(drop=True, inplace=True)

display(results_df)

[KNN] Đang huấn luyện với GridSearchCV...
Fitting 5 folds for each of 60 candidates, totalling 300 fits
{'model__n_neighbors': 7, 'model__p': 1, 'model__weights': 'uniform', 'pca__n_components': 0.85}
[LR] Đang huấn luyện với GridSearchCV...
Fitting 5 folds for each of 15 candidates, totalling 75 fits


c:\Users\admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


{'model__C': 10, 'model__penalty': 'l2', 'model__solver': 'lbfgs', 'pca__n_components': 0.85}
[SVM] Đang huấn luyện với GridSearchCV...
Fitting 5 folds for each of 48 candidates, totalling 240 fits
{'model__C': 0.1, 'model__gamma': 'scale', 'model__kernel': 'rbf', 'pca__n_components': 0.85}
[DT] Đang huấn luyện với GridSearchCV...
Fitting 5 folds for each of 270 candidates, totalling 1350 fits
{'model__criterion': 'entropy', 'model__max_depth': 3, 'model__min_samples_leaf': 4, 'model__min_samples_split': 5, 'pca__n_components': 0.9}
[RF] Đang huấn luyện với GridSearchCV...
Fitting 5 folds for each of 144 candidates, totalling 720 fits
{'model__max_depth': 10, 'model__min_samples_leaf': 1, 'model__min_samples_split': 5, 'model__n_estimators': 100, 'pca__n_components': 0.95}
[Bagging] Đang huấn luyện với GridSearchCV...
Fitting 5 folds for each of 81 candidates, totalling 405 fits
{'model__max_features': 1.0, 'model__max_samples': 0.5, 'model__n_estimators': 200, 'pca__n_components': 0.8

,Model,Training Time (s),AUC,Optimal Threshold,Accuracy,Precision,Recall,Best Params
0,SVM,12.5331,0.826701,0.652145,0.816667,0.750000,0.631579,"{'model__C': 0.1, 'model__gamma': 'scale', 'mo..."
1,RF,78.4374,0.820282,0.492405,0.833333,0.736842,0.736842,"{'model__max_depth': 10, 'model__min_samples_l..."
2,XGB,75.5258,0.818999,0.434009,0.783333,0.625000,0.789474,"{'model__colsample_bytree': 1.0, 'model__learn..."
3,Ada,10.7354,0.798460,0.564701,0.816667,0.722222,0.684211,"{'model__learning_rate': 0.1, 'model__n_estima..."
4,KNN,7.4691,0.795892,0.285714,0.616667,0.450000,0.947368,"{'model__n_neighbors': 7, 'model__p': 1, 'mode..."
5,Bagging,45.0590,0.792683,0.355000,0.750000,0.583333,0.736842,"{'model__max_features': 1.0, 'model__max_sampl..."
6,CatBoost,142.1804,0.785623,0.498030,0.816667,0.750000,0.631579,"{'model__depth': 4, 'model__iterations': 300, ..."
7,LR,2.0469,0.768935,0.512086,0.800000,0.684211,0.684211,"{'model__C': 10, 'model__penalty': 'l2', 'mode..."
8,DT,21.7584,0.765725,0.512195,0.750000,0.583333,0.736842,"{'model__criterion': 'entropy', 'model__max_de..."
